## Import Libraries

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from xgboost import XGBClassifier

## Load Data

In [2]:
def load_data(path):
    df = pd.read_csv(path)
    return df

## Split Data

In [3]:
def split_data(df):
    X = df.drop("default", axis=1)
    y = df["default"]

    return train_test_split(X, y, test_size=0.2, random_state=42)

## Train Models
- Logistic Regression
- Random Forest
- XGBoost

In [4]:
def train_models(X_train, y_train):
    models = {}

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    models['Logistic Regression'] = lr

    # Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    models['Random Forest'] = rf

    # XGBoost
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    models['XGBoost'] = xgb

    return models

## Evaluate Models
- Accuracy
- ROC_AUC

In [5]:
def evaluate_models(models, X_test, y_test):
    results = {}

    for name, model in models.items():
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob)

        print(f"\n{name}")
        print("="*40)
        print("Accuracy:", acc)
        print("ROC-AUC:", roc)
        print(classification_report(y_test, y_pred))

        results[name] = {
            "model": model,
            "accuracy": acc,
            "roc_auc": roc
        }

    return results


## Select Best Model

In [6]:
def get_best_model(results):
    best = max(results.items(), key=lambda x: x[1]['roc_auc'])
    print(f"\nBest Model: {best[0]}")
    return best[1]['model']

## Save Model

In [7]:
def save_model(model, path):
    joblib.dump(model, path)
    print(f"Model saved to {path}")

## Main Pipeline

In [8]:
if __name__ == "__main__":
    # Load data
    df = load_data("../data/processed/final_data.csv")

    # Split
    X_train, X_test, y_train, y_test = split_data(df)

    # Train
    models = train_models(X_train, y_train)

    # Evaluate
    results = evaluate_models(models, X_test, y_test)

    # Best model
    best_model = get_best_model(results)

    # Save
    save_model(best_model, "../bin/models/model.pkl")

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:45:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Logistic Regression
Accuracy: 0.8076666666666666
ROC-AUC: 0.7233900024228022
              precision    recall  f1-score   support

           0       0.82      0.97      0.89      4687
           1       0.69      0.22      0.33      1313

    accuracy                           0.81      6000
   macro avg       0.75      0.60      0.61      6000
weighted avg       0.79      0.81      0.77      6000


Random Forest
Accuracy: 0.8095
ROC-AUC: 0.7341825219924957
              precision    recall  f1-score   support

           0       0.84      0.94      0.88      4687
           1       0.61      0.36      0.45      1313

    accuracy                           0.81      6000
   macro avg       0.72      0.65      0.67      6000
weighted avg       0.79      0.81      0.79      6000


XGBoost
Accuracy: 0.8126666666666666
ROC-AUC: 0.7534565555487127
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      4687
           1       0.63      0.35

**Observation:**\
Recall for default class (1) is low--0.35

That means, out of all actual defaulters, only 35% are caught by the model and 65% are missed.\

**Problem:**\
In credit risk:
- Missing a defaulter = financial loss
- False alarm (predicting default wrongly) = less harmful

So, recall for class 1 is MORE important than accuracy.

## Apply SMOTE
The dataset was imbalanced, and the initial model showed low recall for defaulters. SMOTE was applied to improve the model’s ability to correctly identify high-risk customers.

In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

## Retrain the Models After SMOTE

In [10]:
if __name__ == "__main__":

    # Train
    models = train_models(X_train, y_train)

    # Evaluate
    results = evaluate_models(models, X_test, y_test)

    # Best model
    best_model = get_best_model(results)

    # Save
    save_model(best_model, "../bin/models/modified-model.pkl")

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:46:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Logistic Regression
Accuracy: 0.6941666666666667
ROC-AUC: 0.7242380644491391
              precision    recall  f1-score   support

           0       0.87      0.71      0.78      4687
           1       0.38      0.64      0.48      1313

    accuracy                           0.69      6000
   macro avg       0.63      0.67      0.63      6000
weighted avg       0.77      0.69      0.72      6000


Random Forest
Accuracy: 0.7683333333333333
ROC-AUC: 0.7239327361204388
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      4687
           1       0.47      0.48      0.47      1313

    accuracy                           0.77      6000
   macro avg       0.66      0.66      0.66      6000
weighted avg       0.77      0.77      0.77      6000


XGBoost
Accuracy: 0.7923333333333333
ROC-AUC: 0.7427072596806873
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      4687
           1       0.

**Observation:**
- Recall ↑ (0.35 → 0.45)
- Accuracy ↓ (0.81 → 0.79)
- ROC-AUC ↓ slightly

Recall is still not enough. So we will try customizing the threshold value which is 0.5 by default and train only the best model (XGBoost) now.

In [11]:
def train_xgboost(X_train, y_train):
    models = {}

    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    models['XGBoost'] = xgb

    return models

In [12]:
def evaluate_xgboost(models, X_test, y_test):
    results = {}

    for name, model in models.items():
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        threshold = 0.3
        y_pred_custom = (y_prob >= threshold).astype(int)


        acc = accuracy_score(y_test, y_pred_custom)
        roc = roc_auc_score(y_test, y_prob)

        print(f"\n{name}")
        print("="*40)
        print("Accuracy:", acc)
        print("ROC-AUC:", roc)
        print(classification_report(y_test, y_pred_custom))

        results[name] = {
            "model": model,
            "accuracy": acc,
            "roc_auc": roc
        }

    return results

In [13]:
if __name__ == "__main__":

    # Train
    models = train_xgboost(X_train, y_train)

    # Evaluate
    xgb_results = evaluate_xgboost(models, X_test, y_test)

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:199: UserWarning: [15:46:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost
Accuracy: 0.6848333333333333
ROC-AUC: 0.7427072596806873
              precision    recall  f1-score   support

           0       0.88      0.69      0.77      4687
           1       0.37      0.65      0.48      1313

    accuracy                           0.68      6000
   macro avg       0.63      0.67      0.63      6000
weighted avg       0.77      0.68      0.71      6000



**Observation:**
- Recall ↑ (0.45 → 0.65)
- Accuracy ↓ (0.79 → 0.68)

## Save the Final Model

In [14]:
xgb_model = get_best_model(xgb_results)
save_model(xgb_model, "../bin/models/final-model.pkl")


Best Model: XGBoost
Model saved to ../bin/models/final-model.pkl
